# Week 10: Time Series Analysis

**Foundations of Data Science**

## In this lesson

1. Rows that aren't independent: what changes when order matters
2. Looking at a series: trend, season, irregular, and a regime break
3. Decomposition
4. Autocorrelation: a series' memory of itself
5. Evaluation without cheating: time-based splits and the baselines to beat
6. Models: exponential smoothing and a regression with seasonal terms
7. Forecasts with intervals, and how far ahead to trust them
8. Pitfalls

## The one idea

**In a time series, the best predictor of this week is last week, and the best predictor of this season is last season.** Everything else is a refinement. Any model that can't beat "same as last week" or "same as this week last year" hasn't earned its complexity, and a surprising number of published forecasts can't.

New decision, same clinic. Every winter, flu fills the waiting room. **How many weeks ahead can we forecast the influenza-like-illness (ILI) rate, how well, and when should the clinic add staff?**

Data: CDC's FluView, the national share of outpatient visits for influenza-like illness, one row per week from 2010 to 2025, via the CMU Delphi Epidata API. `wili` is the weighted (population-adjusted) version, in percent.

## Setup

<!-- nb-links -->
[Download this notebook](week_10_time_series.ipynb?download=1){download="week_10_time_series.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/week_10/week_10_time_series.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/week_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("week_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this week's raw data only if it isn't already on disk, then build the processed table
if not Path("data/raw/fluview.parquet").exists():
    !python data_gen.py fetch --name fluview

print("Working directory:", os.getcwd())

## 1. Load, and make time the index

Epidemiological weeks are coded `YYYYWW`. Each MMWR week ends on a Saturday; we attach that date so pandas knows these are ordered, evenly spaced observations. Once the index is a date, resampling, lags, and rolling windows all work. (One trap already: MMWR years sometimes have 53 weeks, and a naive year-week-to-date conversion produces duplicate dates. The cell checks for it.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

raw = pd.read_parquet("data/raw/fluview.parquet")
raw["year"] = raw["epiweek"] // 100
raw["week"] = raw["epiweek"] % 100

# MMWR weeks end on Saturday, and some years have a week 53 (2014 and 2020 here), so ISO-week conversion
# would collide. The series has no gaps, so anchor on the first week's Saturday (2010-01-09) and step 7 days.
assert raw["epiweek"].is_monotonic_increasing and raw["epiweek"].is_unique
raw["date"] = pd.Timestamp("2010-01-09") + pd.to_timedelta(7 * np.arange(len(raw)), unit="D")
ts = raw.set_index("date")["wili"].asfreq("W-SAT")

print("weeks:", len(ts), "  from", ts.index.min().date(), "to", ts.index.max().date(), "  gaps:", ts.isna().sum())
print("53-week years:", raw.loc[raw["week"] == 53, "year"].tolist())
ts.describe().round(2)

## 2. Look at it

Always plot the whole series first. You're looking for four things: a long-run **trend**, a repeating **seasonal** pattern, the leftover **irregular** wiggle, and any **regime break** where the process itself changed.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ts.plot(ax=ax, linewidth=1)
ax.axvspan("2020-03-01", "2021-10-01", color="gray", alpha=0.15)
ax.text(pd.Timestamp("2020-09-15"), ts.max() * 0.9, "COVID:\nflu season\ndidn't happen", ha="center", fontsize=9, color="gray")
ax.set_ylabel("% of visits for ILI (weighted)"); ax.set_xlabel("")
ax.set_title("US outpatient influenza-like illness, weekly, 2010 to 2025")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Seasonal plot: every year on the same axis, by week of season (week 40 = early October = season start)
sea = raw.copy()
sea["season_week"] = (sea["week"] - 40) % 52
sea["season"] = np.where(sea["week"] >= 40, sea["year"], sea["year"] - 1)
fig, ax = plt.subplots(figsize=(11, 4.5))
for s, g in sea.groupby("season"):
    if s < 2010: continue
    color = "red" if s in (2020,) else ("orange" if s == 2017 else "steelblue")
    alpha = 1.0 if s in (2017, 2020) else 0.35
    ax.plot(g["season_week"], g["wili"], color=color, alpha=alpha, linewidth=1.5 if alpha == 1 else 1, label=f"{s}-{str(s+1)[2:]}" if alpha == 1 else None)
ax.set_xlabel("weeks since season start (week 40)"); ax.set_ylabel("% ILI"); ax.legend()
ax.set_title("Every season overlaid: peaks land between weeks 10 and 20 of the season, heights vary 3x")
ax.grid(alpha=0.3); plt.show()

Strong seasonality with a peak around the turn of the year; no obvious trend; one season (2020-21) that simply didn't happen, and a 2017-18 season that was unusually severe. Peak *timing* is fairly stable; peak *height* is not. That split (when vs how much) is the shape of the forecasting problem.

## 3. Decomposition

**STL** (seasonal-trend decomposition using loess) splits the series into three additive parts: trend + seasonal + remainder. It's a description, not a model, but it tells you how big each part is, and the remainder shows you what the seasonal pattern can't explain. ILI is right-skewed (peaks are spiky), so we decompose the log; on the log scale, seasonality becomes multiplicative, which matches how epidemics grow.

In [ ]:
from statsmodels.tsa.seasonal import STL

log_ts = np.log(ts)
stl = STL(log_ts, period=52, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(12, 8)
fig.axes[0].set_title("STL decomposition of log(ILI), period = 52 weeks")
plt.show()

parts = pd.DataFrame({"trend": stl.trend, "seasonal": stl.seasonal, "remainder": stl.resid})
print("share of variance:\n", (parts.var() / parts.var().sum()).round(2).to_string())

Seasonality carries most of the variance; the trend is nearly flat; the remainder is where 2017-18 and 2020-21 live. `robust=True` keeps those years from distorting the seasonal estimate for everyone else.

## 4. Autocorrelation

**Autocorrelation** at lag k is the correlation between the series and itself shifted by k weeks. At lag 1 it's near 1 (this week looks like last week); at lag 52 it's high again (this week looks like this week last year); at lag 26 it's negative (midsummer vs midwinter). The ACF plot shows all lags at once; the PACF shows each lag's *direct* contribution after removing the shorter lags, which is how you decide how many past weeks a model needs.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(log_ts, lags=110, ax=axes[0]); axes[0].set_title("ACF of log(ILI): peaks at lags 52 and 104 are the seasons")
plot_pacf(log_ts, lags=30, ax=axes[1]); axes[1].set_title("PACF: after lag 1 or 2, little direct memory")
plt.show()

The PACF says most of the week-to-week memory is at lag 1 (and a bit at 2): once you know last week, the week before adds little. The ACF's bumps at 52 and 104 say the seasonal memory is strong. A good model needs both: short memory plus a seasonal term.

## 5. Evaluation without cheating

Week 7's random train/test split is wrong here. A random split puts March 2018 in the training set and February 2018 in the test set, so the model "forecasts" a week it has already seen the neighbors of. The rule: **train on the past, test on the future.** Hold out the last complete season (2024-25) and forecast it from everything before.

Then the baselines. Two cost nothing and are hard to beat:

- **Naive**: next week = this week.
- **Seasonal naive**: next week = the same week last year.

Error metric: mean absolute error (MAE) in percentage points of ILI, which the clinic can interpret directly. For a forecast made h weeks ahead, we'll look at MAE as a function of h, because a 1-week forecast and a 10-week forecast are different products.

In [ ]:
from sklearn.metrics import mean_absolute_error

split = pd.Timestamp("2024-09-28")             # last Saturday of September 2024: the 2024-25 season starts the next week
train = ts[:split]
test = ts[split + pd.Timedelta(weeks=1):"2025-09-27"]   # exactly one season, 52 weeks
train_log = np.log(train)
h = len(test)
print(f"train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} weeks)")
print(f"test:  {test.index.min().date()} to {test.index.max().date()} ({h} weeks)")

# Baselines, forecast for the whole test period from the end of training
naive = pd.Series(train.iloc[-1], index=test.index)
seasonal_naive = pd.Series(train.iloc[-52:].values[: h] if h <= 52 else np.tile(train.iloc[-52:].values, 2)[: h], index=test.index)

print("\nMAE over the test season:")
print(f"  naive            {mean_absolute_error(test, naive):.2f}")
print(f"  seasonal naive   {mean_absolute_error(test, seasonal_naive):.2f}")

## 6. Two models

**Holt-Winters exponential smoothing** (ETS): a weighted average of the past where recent weeks count more, with separate smoothed components for level, trend, and season. Each has a smoothing parameter between 0 and 1 saying how fast it forgets: 0 means "never update, keep the initial estimate," 1 means "trust only the latest observation." Fit by minimizing one-step-ahead error. The workhorse for seasonal series; it's what most retail and staffing forecasts run on. The series has no trend, so we leave that component out.

**Regression with seasonal terms**: week 6's tool, with time as a predictor and the season encoded as a few sine and cosine waves (**Fourier terms**) at the annual period. Plus lagged values of the series as predictors, which is how a regression gets memory. It's less standard than ETS but you already know how to read it, and it extends naturally to external predictors (temperature, school calendar, vaccination rates).

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Holt-Winters on the log scale, multiplicative-by-construction once we exponentiate back
hw = ExponentialSmoothing(train_log, trend=None, seasonal="add", seasonal_periods=52).fit()   # no trend term: the series has none
hw_fc = np.exp(hw.forecast(h))
print(f"Holt-Winters MAE: {mean_absolute_error(test, hw_fc):.2f}")
print("smoothing parameters:", {k: round(v, 3) for k, v in hw.params.items() if k in ("smoothing_level", "smoothing_seasonal")})

Look at the fitted parameters before the MAE. Level smoothing of 1.0 and seasonal smoothing of 0.0 means the optimizer concluded: "the best estimate of this week's level is last week's observation, and the seasonal pattern should never be updated from its initial estimate." That is a seasonal naive forecast with a smoothed starting season. The model told you, in its parameters, that there isn't much more structure to extract than the baseline already has; its MAE being a hair better than seasonal naive says the same thing. Parameters at the boundary are always worth reading; they're the model's confession.

In [ ]:
import statsmodels.api as sm

def fourier_frame(index: pd.DatetimeIndex, K: int = 3, period: float = 52.18) -> pd.DataFrame:
    # K pairs of sine/cosine at the annual period and its harmonics; t counts weeks since the start of the series
    t = np.arange(len(index)) + (index[0] - ts.index[0]).days / 7
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(2 * np.pi * k * t / period)
        cols[f"cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=index)

# Regression on log(ILI): Fourier seasonality + lag-1 and lag-52 of the series. Fit on train.
def design(series_log: pd.Series) -> pd.DataFrame:
    X = fourier_frame(series_log.index)
    X["lag1"] = series_log.shift(1)
    X["lag52"] = series_log.shift(52)
    return sm.add_constant(X)

X_train = design(train_log).dropna()
ols = sm.OLS(train_log.loc[X_train.index], X_train).fit()

# Forecast recursively: each predicted week becomes next week's lag1. lag52 comes from actual history (it's a year old, so it's known).
history = train_log.copy()
preds = []
for date in test.index:
    X_row = fourier_frame(pd.DatetimeIndex([date]))
    X_row["lag1"] = history.iloc[-1]
    X_row["lag52"] = history.loc[date - pd.Timedelta(weeks=52)]
    X_row = sm.add_constant(X_row, has_constant="add")[X_train.columns]
    p = float(ols.predict(X_row).iloc[0])
    preds.append(p); history.loc[date] = p
reg_fc = np.exp(pd.Series(preds, index=test.index))
print(f"Regression (Fourier + lags) MAE: {mean_absolute_error(test, reg_fc):.2f}")
print(ols.params.round(3).to_string())

## 7. Forecasts with intervals, and the horizon

A point forecast without a band is a guess (week 3, again). Holt-Winters gives analytic prediction intervals; for the regression we'd bootstrap the residuals. And the error grows with the horizon: a one-week-ahead forecast is mostly "same as last week"; a twelve-week-ahead forecast is mostly "same as last year," which is a much weaker claim.

In [ ]:
hw_sim = hw.simulate(h, repetitions=500, error="add", random_errors="bootstrap")   # bootstrap the fitted residuals
lo, hi = np.exp(hw_sim.quantile(0.05, axis=1)), np.exp(hw_sim.quantile(0.95, axis=1))

fig, ax = plt.subplots(figsize=(13, 4.5))
ts["2022-10-01":].plot(ax=ax, color="black", linewidth=1, label="actual")
hw_fc.plot(ax=ax, color="C0", label="Holt-Winters")
ax.fill_between(test.index, lo, hi, color="C0", alpha=0.15, label="90% band (bootstrap)")
reg_fc.plot(ax=ax, color="C1", label="regression (Fourier + lags)")
seasonal_naive.plot(ax=ax, color="gray", linestyle="--", label="seasonal naive")
ax.axvline(split, color="gray", linewidth=1)
ax.set_ylabel("% ILI"); ax.set_xlabel(""); ax.legend(loc="upper left"); ax.grid(alpha=0.3)
ax.set_title("Forecasting the 2024-25 season from September 2024")
plt.show()

In [ ]:
# Error by horizon: refit nothing, just measure how MAE grows as we look further from the split
by_h = pd.DataFrame({
    "h (weeks ahead)": np.arange(1, h + 1),
    "seasonal naive": np.abs(test.values - seasonal_naive.values),
    "Holt-Winters": np.abs(test.values - hw_fc.values),
    "regression": np.abs(test.values - reg_fc.values),
}).set_index("h (weeks ahead)")
by_h.rolling(4).mean().plot(figsize=(9, 4), title="Absolute error by horizon (4-week rolling mean)", grid=True, ylabel="|error|, % ILI")
plt.show()
print("MAE by horizon block:")
print(by_h.groupby(pd.cut(by_h.index, [0, 4, 12, 26, 52], labels=["1-4", "5-12", "13-26", "27-52"])).mean().round(2))

Read the horizon table against the plot. One to four weeks out, both models cut the seasonal-naive error by more than half (0.15 to 0.20 against 0.41): short memory works. Five to twelve weeks out, through the ramp, they still halve it. Thirteen to twenty-six weeks out is the peak, and nobody does well: 2024-25 had a second, higher peak in February that the previous year didn't, and a September forecast can't know that. Past the peak everything converges because summer is always low.

The number to report to the clinic is MAE over the horizon they actually plan on. If staffing is decided four weeks out, the four-week error is the only one that matters, and it's small. If they want to know in September how bad February will be, the honest answer is the 90% band, which spans 3% to 8% at the peak.

What none of the models can do: predict that a given season will be a 2017-18, a 2020-21, or a double-peaked 2024-25. Peak height and shape depend on which strains circulate and on things that happen *during* the season. A one-season-ahead forecast made in September is a forecast of the average season, with a band wide enough to admit it.

## 8. Pitfalls

- **Random splits** on time series. Train on the past, test on the future, always.
- **Evaluating in-sample.** A seasonal model fit to 15 years will trace those 15 years beautifully. Only held-out seasons count.
- **Ignoring the baseline.** Report seasonal naive next to your model. If you don't beat it, say so.
- **Regime breaks.** 2020-21 is not a data error; it's a year the process changed. Decide explicitly whether it belongs in training (robust STL, or drop it, or model it with a dummy), and say which you did.
- **Long horizons.** Error compounds. State the horizon with every number.
- **Calendar effects.** Week 53 exists in some years; holidays shift clinic volume. Weekly data mostly dodges this; daily data doesn't.

## Looking ahead
- **Reading**: Hyndman & Athanasopoulos chapters 5 (forecaster's toolbox) and 8 (exponential smoothing). For next week: Easley & Kleinberg, *Networks, Crowds, and Markets*, chapters 2 and 3 (free online). The rows become nodes and the structure is the data.